In [12]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [13]:
!pip install -q mlflow dagshub

import mlflow
import dagshub

dagshub.init(repo_owner='aochi23', repo_name='ml_assn_02', mlflow=True)
mlflow.set_experiment('RandomForest_Training')

Initialized MLflow to track repo "aochi23/ml_assn_02"

Repository aochi23/ml_assn_02 initialized!

<Experiment: artifact_location='mlflow-artifacts:/365154bf6230404fa936f7c0bcafab93', creation_time=1777741445533, experiment_id='3', last_update_time=1777741445533, lifecycle_stage='active', name='RandomForest_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

# 1. Data Loading

In [3]:
path = '/kaggle/input/competitions/ieee-fraud-detection/'

train_transaction = pd.read_csv(path + 'train_transaction.csv')
train_identity = pd.read_csv(path + 'train_identity.csv')

In [4]:
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
print(f'Train shape: {train.shape}')

Train shape: (590540, 434)


In [5]:
fraud_counts  = train['isFraud'].value_counts()
fraud_rate    = train['isFraud'].mean() * 100
neg_pos_ratio = int((1 - train['isFraud'].mean()) / train['isFraud'].mean())

print(fraud_counts)
print(f'\nFraud rate: {fraud_rate:.2f}%')
print(f'class_weight neg/pos ratio: {neg_pos_ratio}')

isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud rate: 3.50%
class_weight neg/pos ratio: 27


# 2. Cleaning

In [6]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from joblib import Parallel, delayed
import mlflow.sklearn
import copy
import warnings
warnings.filterwarnings('ignore')

In [7]:
class DropIDTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return X.drop(columns=['TransactionID'], errors='ignore').reset_index(drop=True)

In [8]:
class EmailMatchTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        if 'P_emaildomain' in X.columns and 'R_emaildomain' in X.columns:
            X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype(np.int8)
        else:
            X['email_match'] = 0
        return X

In [9]:
class MissingFlagTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.high_missing_columns = None

    def fit(self, X, y=None):
        missing_rate = X.isnull().mean()
        self.high_missing_columns = missing_rate[missing_rate > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        missing_flags = X[self.high_missing_columns].isnull().astype(np.int8)
        missing_flags.columns = [f'{col}_missing' for col in self.high_missing_columns]
        return pd.concat([X.drop(columns=self.high_missing_columns), missing_flags], axis=1)

In [10]:
class LabelEncodeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.cat_cols = None

    def fit(self, X, y=None):
        self.cat_cols = X.select_dtypes(include=['object']).columns.tolist()
        for col in self.cat_cols:
            le = LabelEncoder()
            le.fit(X[col].fillna('unknown').astype(str))
            self.encoders[col] = le
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols:
            X[col] = X[col].fillna('unknown').astype(str)
            known  = set(self.encoders[col].classes_)
            X[col] = X[col].apply(lambda v: v if v in known else 'unknown')
            X[col] = self.encoders[col].transform(X[col])
        return X

In [11]:
class TargetEncodeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, smoothing=10):
        self.cols         = cols or ['card1', 'card2', 'addr1', 'P_emaildomain', 'R_emaildomain']
        self.smoothing    = smoothing
        self.global_mean  = None
        self.encoding_map = {}

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y).reset_index(drop=True)
        X = X.reset_index(drop=True)
        self.global_mean = y.mean()
        for col in self.cols:
            if col not in X.columns:
                continue
            X[col] = X[col].fillna('unknown').astype(str)
            stats  = pd.DataFrame({'target': y, 'col': X[col]})
            agg    = stats.groupby('col')['target'].agg(['mean', 'count'])
            smooth = (agg['count'] * agg['mean'] + self.smoothing * self.global_mean) / \
                     (agg['count'] + self.smoothing)
            self.encoding_map[col] = smooth
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            if col not in X.columns:
                continue
            X[col] = X[col].fillna('unknown').astype(str)
            X[col] = X[col].map(self.encoding_map[col]).fillna(self.global_mean)
        return X

In [12]:
class RichAggregationTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.card_stats      = {}
        self.user_time_stats = None

    def fit(self, X, y=None):
        X = X.copy()
        for col in ['card1', 'card2']:
            self.card_stats[col] = (
                X.groupby(col)['TransactionAmt']
                .agg(['mean', 'std', 'count'])
                .rename(columns={
                    'mean':  f'{col}_mean_amt',
                    'std':   f'{col}_std_amt',
                    'count': f'{col}_count'
                })
            )
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str)
        )
        self.user_time_stats = (
            X.groupby('user_id')['TransactionDT']
            .agg(['mean', 'std'])
            .rename(columns={'mean': 'user_dt_mean', 'std': 'user_dt_std'})
        )
        return self

    def transform(self, X):
        X = X.copy()
        for col in ['card1', 'card2']:
            X = X.join(self.card_stats[col], on=col)
            X[f'{col}_std_amt'] = X[f'{col}_std_amt'].fillna(0)
            X[f'{col}_amt_zscore'] = (
                (X['TransactionAmt'] - X[f'{col}_mean_amt']) /
                X[f'{col}_std_amt'].replace(0, 1)
            ).clip(-5, 5)
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str)
        )
        X = X.join(self.user_time_stats, on='user_id')
        X['user_dt_std']      = X['user_dt_std'].fillna(0)
        X['amt_is_round']     = (X['TransactionAmt'] % 1 == 0).astype(int)
        X['amt_is_round_100'] = (X['TransactionAmt'] % 100 == 0).astype(int)
        X['addr_mismatch']    = (X['addr1'] != X['addr2']).astype(int)
        X = X.drop(columns=['user_id'])
        return X

In [13]:
class VColumnPCATransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=50):
        self.n_components = n_components
        self.pca     = None
        self.imputer = None
        self.v_cols  = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.v_cols  = [c for c in X.columns if str(c).startswith('V')]
        self.imputer = SimpleImputer(strategy='median')
        self.pca     = PCA(n_components=self.n_components, random_state=42)
        v_data       = self.imputer.fit_transform(X[self.v_cols])
        self.pca.fit(v_data)
        return self

    def transform(self, X):
        X      = X.copy()
        v_data = self.imputer.transform(X[self.v_cols])
        pca_df = pd.DataFrame(
            self.pca.transform(v_data),
            columns=[f'V_pca_{i}' for i in range(self.n_components)],
            index=X.index
        )
        X = X.drop(columns=self.v_cols)
        return pd.concat([X, pca_df], axis=1)

In [14]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.user_stats = None

    def fit(self, X, y=None):
        X = X.copy()
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str) + '_' +
            X['P_emaildomain'].fillna('unknown').astype(str)
        )
        self.user_stats = (
            X.groupby('user_id')['TransactionAmt']
            .agg(['count', 'mean', 'std', 'max'])
            .rename(columns={'count': 'uid_count', 'mean': 'uid_mean',
                             'std':   'uid_std',   'max':  'uid_max'})
        )
        return self

    def transform(self, X):
        X = X.copy()
        X['hour'] = (X['TransactionDT'] // 3600) % 24
        X['day']  = (X['TransactionDT'] // (3600 * 24)) % 7
        X['TransactionAmt_log']   = np.log1p(X['TransactionAmt'])
        X['TransactionAmt_cents'] = X['TransactionAmt'] % 1
        X['user_id'] = (
            X['card1'].fillna(-1).astype(str) + '_' +
            X['card2'].fillna(-1).astype(str) + '_' +
            X['addr1'].fillna(-1).astype(str) + '_' +
            X['P_emaildomain'].fillna('unknown').astype(str)
        )
        X = X.join(self.user_stats, on='user_id')
        X = X.rename(columns={
            'uid_count': 'user_id_count', 'uid_mean': 'user_id_mean_amt',
            'uid_std':   'user_id_std_amt', 'uid_max': 'user_id_max_amt'
        })
        X['user_id_std_amt']   = X['user_id_std_amt'].fillna(0)
        X['user_amt_zscore']   = ((X['TransactionAmt'] - X['user_id_mean_amt']) / X['user_id_std_amt'].replace(0, 1)).clip(-5, 5)
        X['user_amt_ratio']    = (X['TransactionAmt'] / X['user_id_mean_amt']).clip(0, 20)
        X['user_id_count_log'] = np.log1p(X['user_id_count'])
        X['amt_vs_max']        = (X['TransactionAmt'] / X['user_id_max_amt']).clip(0, 1)
        X['user_time_delta']   = X.groupby('user_id')['TransactionDT'].transform(lambda x: x.diff().fillna(0))
        X = X.drop(columns=['user_id'])
        return X

In [16]:
X_raw = train_transaction.merge(train_identity, on='TransactionID', how='left')
y     = X_raw['isFraud']
X_raw = X_raw.drop(columns=['isFraud'])

with mlflow.start_run(run_name='RF_Cleaning'):
    mlflow.log_params({
        'missing_threshold':         0.5,
        'email_match_feature_added': True,
        'label_encoding':            True,
        'target_encoding_smoothing': 10,
        'vpca_components':           50,
    })
    mlflow.log_metrics({
        'total_rows':    X_raw.shape[0],
        'total_columns': X_raw.shape[1],
    })

🏃 View run RF_Cleaning at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3/runs/d7c482f6cce04d729202b24a6c33220a
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3


# 3. Feature Engineering

In [17]:
with mlflow.start_run(run_name='RF_Feature_Engineering'):
    mlflow.log_params({
        'user_id_fields': 'card1_card2_addr1_P_emaildomain',
        'new_features':   'hour,day,TransactionAmt_log,TransactionAmt_cents,user_id_count,user_id_mean_amt,user_id_std_amt,user_id_max_amt,user_amt_zscore,user_amt_ratio,user_id_count_log,amt_vs_max,user_time_delta,card1_aggs,card2_aggs,amt_is_round,amt_is_round_100,addr_mismatch',
        'zscore_clip':    '(-5, 5)',
        'ratio_clip':     '(0, 20)',
    })

🏃 View run RF_Feature_Engineering at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3/runs/3fef303b0d7b4ce1832fcd8cdca5c132
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3


# 4. Feature Selection

In [20]:
class IVSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.02, sample_size=50_000, random_state=42, n_jobs=-1):
        self.threshold        = threshold
        self.sample_size      = sample_size
        self.random_state     = random_state
        self.n_jobs           = n_jobs
        self.iv_scores        = {}
        self.selected_features = None
        self.median           = None

    def compute_iv(self, X, y_binary, col):
        series = X[col].copy()
        if series.nunique() > 10:
            series = pd.qcut(series, q=10, duplicates='drop')
        df = pd.DataFrame({'col': series, 'target': y_binary})
        total_e  = y_binary.sum()
        total_ne = (1 - y_binary).sum()
        grp = df.groupby('col', observed=True)['target'].agg(['sum', 'count'])
        grp.columns = ['e', 'cnt']
        grp['ne'] = grp['cnt'] - grp['e']
        dist_e  = (grp['e']  + 0.5) / (total_e  + 0.5)
        dist_ne = (grp['ne'] + 0.5) / (total_ne + 0.5)
        return ((dist_e - dist_ne) * np.log(dist_e / dist_ne)).sum()

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        X.columns = [str(c) for c in X.columns]
        y = pd.Series(y).reset_index(drop=True)
        if self.sample_size and len(X) > self.sample_size:
            idx      = y.sample(self.sample_size, random_state=self.random_state).index
            X_sample = X.loc[idx].reset_index(drop=True)
            y_sample = y.loc[idx].reset_index(drop=True)
        else:
            X_sample, y_sample = X, y
        if set(y_sample.unique()).issubset({0, 1}):
            y_binary = y_sample.astype(int)
        else:
            self.median = y_sample.median()
            y_binary    = (y_sample > self.median).astype(int)
        results = Parallel(n_jobs=self.n_jobs)(
            delayed(self.compute_iv)(X_sample, y_binary, col)
            for col in X_sample.columns
        )
        self.iv_scores         = dict(zip(X_sample.columns, results))
        self.selected_features = [col for col, iv in self.iv_scores.items() if iv >= self.threshold]
        return self

    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        X.columns = [str(c) for c in X.columns]
        return X[self.selected_features]

In [21]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.85, sample_size=50_000, random_state=42):
        self.threshold = threshold
        self.sample_size = sample_size      
        self.random_state = random_state    
        self.features_to_drop = None
        self.selected_features = None

    def fit(self, X, y=None):
        X = pd.DataFrame(X).reset_index(drop=True)
        X.columns = [str(c) for c in X.columns]

        if self.sample_size and len(X) > self.sample_size:
            X_sample = X.sample(self.sample_size, random_state=self.random_state
                                ).reset_index(drop=True)
        else:
            X_sample = X

        corr_matrix = X_sample.corr().abs()  
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )

        self.features_to_drop = set()
        for col in upper.columns:
            if any(upper[col] > self.threshold):
                for partner in upper.index[upper[col] > self.threshold].tolist():
                    if corr_matrix[col].mean() >= corr_matrix[partner].mean():
                        self.features_to_drop.add(col)
                    else:
                        self.features_to_drop.add(partner)

        self.selected_features = [
            col for col in X.columns if col not in self.features_to_drop
        ]
        return self

    def transform(self, X):
        X = pd.DataFrame(X).reset_index(drop=True)
        X.columns = [str(c) for c in X.columns]
        return X[self.selected_features]

In [22]:
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print(f'Train: {X_train_raw.shape} | Val: {X_val_raw.shape}')

Train: (472432, 433) | Val: (118108, 433)


In [23]:
def make_preprocessing():
    return [
        ('drop_id',    DropIDTransformer()),
        ('email',      EmailMatchTransformer()),
        ('missing',    MissingFlagTransformer(threshold=0.5)),
        ('rich_agg',   RichAggregationTransformer()),
        ('target_enc', TargetEncodeTransformer()),
        ('encode',     LabelEncodeTransformer()),
        ('features',   FeatureEngineeringTransformer()),
        ('vpca',       VColumnPCATransformer(n_components=50)),
    ]

base_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=2
)

## 4a. Strategy 1 — No Feature Selection (baseline)

In [24]:
pipeline_no_sel = Pipeline(make_preprocessing() + [
    ('model', copy.deepcopy(base_rf))
])
pipeline_no_sel.fit(X_train_raw, y_train)

auc_no_sel       = roc_auc_score(y_val,   pipeline_no_sel.predict_proba(X_val_raw)[:, 1])
train_auc_no_sel = roc_auc_score(y_train, pipeline_no_sel.predict_proba(X_train_raw)[:, 1])
n_feats_no_sel   = pipeline_no_sel[:-1].transform(X_train_raw.head(100)).shape[1]

print(f'Strategy 1 | Train: {train_auc_no_sel:.4f} | Val: {auc_no_sel:.4f} | Gap: {train_auc_no_sel - auc_no_sel:.4f} | Features: {n_feats_no_sel}')

with mlflow.start_run(run_name='RF_FeatureSelection_NoSelection'):
    mlflow.log_param('strategy',        'no_selection')
    mlflow.log_param('n_features',      n_feats_no_sel)
    mlflow.log_metric('train_auc',      train_auc_no_sel)
    mlflow.log_metric('val_auc',        auc_no_sel)
    mlflow.log_metric('overfit_gap',    train_auc_no_sel - auc_no_sel)

Strategy 1 | Train: 0.9463 | Val: 0.9255 | Gap: 0.0209 | Features: 170
🏃 View run RF_FeatureSelection_NoSelection at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3/runs/1e937e9574ed4cc7af23816f648ffd26
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3


## 4b. Strategy 2 — IV Selection only

In [25]:
pipeline_iv = Pipeline(make_preprocessing() + [
    ('iv',    IVSelector(threshold=0.02, sample_size=50_000)),
    ('model', copy.deepcopy(base_rf))
])
pipeline_iv.fit(X_train_raw, y_train)

auc_iv       = roc_auc_score(y_val,   pipeline_iv.predict_proba(X_val_raw)[:, 1])
train_auc_iv = roc_auc_score(y_train, pipeline_iv.predict_proba(X_train_raw)[:, 1])
n_feats_iv   = len(pipeline_iv.named_steps['iv'].selected_features)

print(f'Strategy 2 | Train: {train_auc_iv:.4f} | Val: {auc_iv:.4f} | Gap: {train_auc_iv - auc_iv:.4f} | Features: {n_feats_iv}')

with mlflow.start_run(run_name='RF_FeatureSelection_IVOnly'):
    mlflow.log_param('strategy',           'iv_selector')
    mlflow.log_param('iv_threshold',       0.02)
    mlflow.log_param('iv_sample_size',     50_000)
    mlflow.log_param('n_features_kept',    n_feats_iv)
    mlflow.log_param('n_features_dropped', n_feats_no_sel - n_feats_iv)
    mlflow.log_metric('train_auc',         train_auc_iv)
    mlflow.log_metric('val_auc',           auc_iv)
    mlflow.log_metric('overfit_gap',       train_auc_iv - auc_iv)

Strategy 2 | Train: 0.9462 | Val: 0.9250 | Gap: 0.0212 | Features: 160
🏃 View run RF_FeatureSelection_IVOnly at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3/runs/6bc3c08506bf4843a00abc999796a3ff
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3


## 4c. Strategy 3 — IV Selection + Correlation Filter (combined)

In [26]:
pipeline_iv_corr = Pipeline(make_preprocessing() + [
    ('iv',    IVSelector(threshold=0.02, sample_size=50_000)),
    ('corr',  CorrelationFilter(threshold=0.85, sample_size=50_000)),
    ('model', copy.deepcopy(base_rf))
])
pipeline_iv_corr.fit(X_train_raw, y_train)

auc_iv_corr       = roc_auc_score(y_val,   pipeline_iv_corr.predict_proba(X_val_raw)[:, 1])
train_auc_iv_corr = roc_auc_score(y_train, pipeline_iv_corr.predict_proba(X_train_raw)[:, 1])
n_feats_iv_corr   = len(pipeline_iv_corr.named_steps['corr'].selected_features)

print(f'Strategy 3 | Train: {train_auc_iv_corr:.4f} | Val: {auc_iv_corr:.4f} | Gap: {train_auc_iv_corr - auc_iv_corr:.4f} | Features: {n_feats_iv_corr}')

with mlflow.start_run(run_name='RF_FeatureSelection_IV_and_Corr'):
    mlflow.log_param('strategy',           'iv_selector_plus_corr_filter')
    mlflow.log_param('iv_threshold',       0.02)
    mlflow.log_param('corr_threshold',     0.85)
    mlflow.log_param('iv_sample_size',     50_000)
    mlflow.log_param('corr_sample_size',   50_000)
    mlflow.log_param('n_features_kept',    n_feats_iv_corr)
    mlflow.log_param('n_features_dropped', n_feats_no_sel - n_feats_iv_corr)
    mlflow.log_metric('train_auc',         train_auc_iv_corr)
    mlflow.log_metric('val_auc',           auc_iv_corr)
    mlflow.log_metric('overfit_gap',       train_auc_iv_corr - auc_iv_corr)

Strategy 3 | Train: 0.9424 | Val: 0.9154 | Gap: 0.0270 | Features: 101
🏃 View run RF_FeatureSelection_IV_and_Corr at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3/runs/ff5c9d0eeff34152afe4dee20aa1509f
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3


In [27]:
print(f'No Selection  | Val: {auc_no_sel:.4f} | Gap: {train_auc_no_sel - auc_no_sel:.4f} | Features: {n_feats_no_sel}')
print(f'IV Only       | Val: {auc_iv:.4f} | Gap: {train_auc_iv - auc_iv:.4f} | Features: {n_feats_iv}')
print(f'IV + Corr     | Val: {auc_iv_corr:.4f} | Gap: {train_auc_iv_corr - auc_iv_corr:.4f} | Features: {n_feats_iv_corr}')

No Selection  | Val: 0.9255 | Gap: 0.0209 | Features: 170
IV Only       | Val: 0.9250 | Gap: 0.0212 | Features: 160
IV + Corr     | Val: 0.9154 | Gap: 0.0270 | Features: 101


# 5. Training — Hyperparameter Tuning

In [36]:
tuning_pipeline = Pipeline(make_preprocessing() + [
    ('model', RandomForestClassifier(
        class_weight='balanced',
        random_state=42,
        n_jobs=2
    ))
])

param_grid_rf = {
    'model__n_estimators':     [300],
    'model__max_depth':        [8, 12],
    'model__min_samples_leaf': [1, 5],
}

In [31]:
def run_and_log_rf(run_name, pipeline, param_grid):
    with mlflow.start_run(run_name=run_name):
        grid = GridSearchCV(
            pipeline, param_grid, cv=skf,
            scoring='roc_auc', n_jobs=2,
            return_train_score=True,
            verbose=2
        )
        grid.fit(X_train_raw, y_train)

        for i, params in enumerate(grid.cv_results_['params']):
            run_label = '_'.join([
                f"depth{params['model__max_depth']}",
                f"n{params['model__n_estimators']}",
                f"leaf{params['model__min_samples_leaf']}",
            ])
            with mlflow.start_run(run_name=f'RF_CV_{run_label}', nested=True):
                mlflow.log_params(params)
                mlflow.log_metric('cv_auc_mean',       grid.cv_results_['mean_test_score'][i])
                mlflow.log_metric('cv_auc_std',        grid.cv_results_['std_test_score'][i])
                mlflow.log_metric('cv_train_auc_mean', grid.cv_results_['mean_train_score'][i])
                mlflow.log_metric('overfit_gap',
                    grid.cv_results_['mean_train_score'][i] - grid.cv_results_['mean_test_score'][i]
                )

        best_pipeline = grid.best_estimator_
        train_auc = roc_auc_score(y_train, best_pipeline.predict_proba(X_train_raw)[:, 1])
        val_auc   = roc_auc_score(y_val,   best_pipeline.predict_proba(X_val_raw)[:, 1])

        mlflow.log_params(grid.best_params_)
        mlflow.log_metric('cv_auc_mean',  grid.best_score_)
        mlflow.log_metric('cv_auc_std',   grid.cv_results_['std_test_score'][grid.best_index_])
        mlflow.log_metric('train_auc',    train_auc)
        mlflow.log_metric('val_auc',      val_auc)
        mlflow.log_metric('overfit_gap',  train_auc - val_auc)

        print(f'{run_name} | CV: {grid.best_score_:.4f} | Train: {train_auc:.4f} | Val: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}')
    return grid

In [32]:
grid_rf = run_and_log_rf('RF_CV', tuning_pipeline, param_grid_rf)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END model__max_depth=8, model__min_samples_leaf=1, model__n_estimators=200; total time= 4.9min
[CV] END model__max_depth=8, model__min_samples_leaf=1, model__n_estimators=200; total time= 5.1min
[CV] END model__max_depth=8, model__min_samples_leaf=1, model__n_estimators=400; total time= 9.1min
[CV] END model__max_depth=8, model__min_samples_leaf=5, model__n_estimators=200; total time= 5.3min
[CV] END model__max_depth=8, model__min_samples_leaf=5, model__n_estimators=200; total time= 5.0min
[CV] END model__max_depth=8, model__min_samples_leaf=5, model__n_estimators=400; total time= 9.7min
[CV] END model__max_depth=12, model__min_samples_leaf=1, model__n_estimators=200; total time= 6.7min
[CV] END model__max_depth=12, model__min_samples_leaf=1, model__n_estimators=200; total time= 6.6min
[CV] END model__max_depth=12, model__min_samples_leaf=1, model__n_estimators=400; total time=12.1min
[CV] END model__max_depth=12, model_

KeyboardInterrupt: 

In [37]:
grid_rf = run_and_log_rf('RF_CV', tuning_pipeline, param_grid_rf)

Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END model__max_depth=8, model__min_samples_leaf=1, model__n_estimators=300; total time= 7.3min
[CV] END model__max_depth=8, model__min_samples_leaf=5, model__n_estimators=300; total time= 7.2min
[CV] END model__max_depth=8, model__min_samples_leaf=5, model__n_estimators=300; total time= 7.2min
[CV] END model__max_depth=12, model__min_samples_leaf=1, model__n_estimators=300; total time= 9.2min
[CV] END model__max_depth=8, model__min_samples_leaf=1, model__n_estimators=300; total time= 7.1min
[CV] END model__max_depth=8, model__min_samples_leaf=1, model__n_estimators=300; total time= 7.5min
[CV] END model__max_depth=8, model__min_samples_leaf=5, model__n_estimators=300; total time= 7.0min
[CV] END model__max_depth=12, model__min_samples_leaf=1, model__n_estimators=300; total time= 9.0min
[CV] END model__max_depth=12, model__min_samples_leaf=1, model__n_estimators=300; total time= 9.1min
[CV] END model__max_depth=12, model__

In [38]:
best_params = {k.replace('model__', ''): v for k, v in grid_rf.best_params_.items()}
print(f'Best params: {best_params}')

Best params: {'max_depth': 12, 'min_samples_leaf': 5, 'n_estimators': 300}


# 6. Final Pipeline

In [39]:
best_rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=2,
    **best_params
)

full_pipeline_rf = Pipeline(make_preprocessing() + [
    ('model', best_rf)
])


full_pipeline_rf.fit(X_train_raw, y_train)
val_auc   = roc_auc_score(y_val,   full_pipeline_rf.predict_proba(X_val_raw)[:, 1])
train_auc = roc_auc_score(y_train, full_pipeline_rf.predict_proba(X_train_raw)[:, 1])

print(f'Final RF | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | Gap: {train_auc - val_auc:.4f}')

Final RF | Train AUC: 0.9667 | Val AUC: 0.9347 | Gap: 0.0321


In [40]:
with mlflow.start_run(run_name='RF_Final_Pipeline'):
    mlflow.log_params({
        'missing_threshold': 0.5,
        'vpca_components':   50,
        'feature_selection': 'no_selection',
        'class_weight':      'balanced',
        **best_params
    })
    mlflow.log_metric('train_auc',   train_auc)
    mlflow.log_metric('val_auc',     val_auc)
    mlflow.log_metric('overfit_gap', train_auc - val_auc)
    mlflow.sklearn.log_model(
        full_pipeline_rf,
        artifact_path='full_pipeline_rf',
        registered_model_name='RF_FullPipeline'
    )
    print(f'Final RF pipeline VAL AUC: {val_auc:.4f}')

2026/05/02 22:58:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 22:58:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'RF_FullPipeline'.
2026/05/02 22:59:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF_FullPipeline, version 1
Created version '1' of model 'RF_FullPipeline'.


Final RF pipeline VAL AUC: 0.9347
🏃 View run RF_Final_Pipeline at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3/runs/676cd645cdf84a5699f0e45ae0f43dc9
🧪 View experiment at: https://dagshub.com/aochi23/ml_assn_02.mlflow/#/experiments/3


In [3]:
model = mlflow.pyfunc.load_model("models:/RF_FullPipeline/1")

In [14]:
sk_model = model._model_impl.sklearn_model

print(sk_model)
for k, v in sk_model.get_params().items():
    print(k, ":", v)

Pipeline(steps=[('drop_id', DropIDTransformer()),
                ('email', EmailMatchTransformer()),
                ('missing', MissingFlagTransformer()),
                ('rich_agg', RichAggregationTransformer()),
                ('target_enc',
                 TargetEncodeTransformer(cols=['card1', 'card2', 'addr1',
                                               'P_emaildomain',
                                               'R_emaildomain'])),
                ('encode', LabelEncodeTransformer()),
                ('features', FeatureEngineeringTransformer()),
                ('vpca', VColumnPCATransformer()),
                ('model',
                 RandomForestClassifier(class_weight='balanced', max_depth=12,
                                        min_samples_leaf=5, n_estimators=300,
                                        n_jobs=2, random_state=42))])
memory : None
steps : [('drop_id', DropIDTransformer()), ('email', EmailMatchTransformer()), ('missing', MissingFlagTransforme

In [5]:
path = '/kaggle/input/competitions/ieee-fraud-detection/'

test_transaction = pd.read_csv(path + "test_transaction.csv")
test_identity    = pd.read_csv(path + "test_identity.csv")

In [6]:
print(test_identity.shape)
print(test_identity.columns[:10])

(141907, 41)
Index(['TransactionID', 'id-01', 'id-02', 'id-03', 'id-04', 'id-05', 'id-06',
       'id-07', 'id-08', 'id-09'],
      dtype='object')


In [7]:
test_df = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
)

test_df.columns = test_df.columns.str.replace('-', '_')

In [8]:
probs = sk_model.predict_proba(test_df)[:, 1]

In [9]:
submission = pd.DataFrame({
    "TransactionID": test_df["TransactionID"],
    "isFraud": probs
})

In [10]:
submission.head(10)

,TransactionID,isFraud
0,3663549,0.064324
1,3663550,0.222929
2,3663551,0.081765
3,3663552,0.059528
4,3663553,0.091906
5,3663554,0.098239
6,3663555,0.185140
7,3663556,0.408238
8,3663557,0.061651
9,3663558,0.234824


In [11]:
submission.to_csv("random_forest_submission.csv", index=False)